# 02: Evaluation of the video pipeline

Evaluation of BreathCam v1 using only the data in `data/breathcam.db` (5 clips, one subject, no accelerometer). Reference = **metronome target**, not an independent measurement, so everything below is agreement with a paced target.

Sections: method comparison, conditions, confidence intervals, Bland-Altman, failure analysis.

In [1]:
import sys; sys.path.insert(0, "..")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from breathcam import db
from breathcam.eval import (load_windows, summarize, rejection_rate, bootstrap_ci, plot_bland_altman,
                            bland_altman, mae, rmse)
from breathcam.run import estimate
from breathcam.signal import bandpass, detrend, peak_rate
from breathcam.build_db import ROI

conn = db.connect("../data/breathcam.db")
df = load_windows(conn)
print(len(df), "rows;", df.session_id.nunique(), "sessions;", df.subject_id.nunique(), "subject(s);",
      df.groupby("method").size().to_dict())

99 rows; 5 sessions; 1 subject(s); {'framediff': 33, 'peaks': 33, 'welch': 33}


## 1. Method comparison (scored against the metronome)

In [2]:
summarize(df, ["method", "ref_source"]).round(2)

,,n,mae,rmse,within_2,bias
method,ref_source,,,,,
framediff,metronome,26,0.11,0.16,1.0,0.03
peaks,metronome,26,0.18,0.23,1.0,-0.04
welch,metronome,26,0.12,0.18,1.0,0.03


## 2. Conditions

Only the paced rate varies across these clips. Lighting, clothing, distance and mount are identical or unrecorded, so those tables have one row and say nothing yet.

In [3]:
display(summarize(df, ["paced_bpm", "method"]).round(2))
display(summarize(df, ["lighting", "method"]).round(2))
print("Windows rejected by the quality gate, per clip (welch):")
rejection_rate(df[df.method == "welch"], ["session_id", "video_path"]).round(2)

n   mae  rmse  within_2  bias
paced_bpm method                                   
12.0      framediff  13  0.09  0.14       1.0  0.01
          peaks      13  0.13  0.16       1.0 -0.07
          welch      13  0.10  0.18       1.0 -0.01
20.0      framediff  13  0.13  0.18       1.0  0.06
          peaks      13  0.24  0.29       1.0 -0.01
          welch      13  0.13  0.18       1.0  0.06

n   mae  rmse  within_2  bias
lighting method                                   
NaN      framediff  26  0.11  0.16       1.0  0.03
         peaks      26  0.18  0.23       1.0 -0.04
         welch      26  0.12  0.18       1.0  0.03

Windows rejected by the quality gate, per clip (welch):


,,n,rejected,rejected_frac
session_id,video_path,,,
1,data/raw/01_12bpm.mov,13,0,0.0
2,data/raw/02_20bpm.mov,13,0,0.0
3,data/raw/03_hold.mov,5,2,0.4
4,data/raw/04_wave.mov,1,1,1.0
5,data/raw/05_normal.mov,1,0,0.0


## 3. Confidence intervals (resampling subjects)

Windows from one person are correlated, so the bootstrap resamples whole subjects. With fewer than 5 subjects it refuses to give an interval.

In [4]:
for m in ["welch", "peaks", "framediff"]:
    d = df[(df.method == m) & (df.ref_source == "metronome")]
    point, lo, hi = bootstrap_ci(d, mae, cluster="subject_id")
    print(f"{m:10s} MAE {point:.2f}   95% CI: {'not estimable' if np.isnan(lo) else f'{lo:.2f} to {hi:.2f}'}"
          f"   ({d.subject_id.nunique()} subject, {d.session_id.nunique()} paced clips)")

welch      MAE 0.12   95% CI: not estimable   (1 subject, 2 paced clips)
peaks      MAE 0.18   95% CI: not estimable   (1 subject, 2 paced clips)
framediff  MAE 0.11   95% CI: not estimable   (1 subject, 2 paced clips)


## 4. Bland-Altman (estimate vs metronome target)

In [5]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, m in zip(axes, ["welch", "peaks", "framediff"]):
    d = df[(df.method == m) & df.est_bpm.notna() & df.ref_bpm.notna()]
    plot_bland_altman(d.est_bpm, d.ref_bpm, ax, title=f"{m} vs metronome (n={len(d)})")
    ax.set_xlim(10, 22)
    _, _, bias, lo, hi = bland_altman(d.est_bpm, d.ref_bpm)
    print(f"{m:10s} bias {bias:+.2f}   limits of agreement {lo:+.2f} to {hi:+.2f}")
plt.tight_layout(); plt.savefig("../docs/figures/bland_altman_video_vs_metronome.png", dpi=150)

welch      bias +0.03   limits of agreement -0.34 to +0.39
peaks      bias -0.04   limits of agreement -0.50 to +0.42
framediff  bias +0.03   limits of agreement -0.28 to +0.35


## 5. Failure analysis

### 5a. Largest errors among accepted, scored windows (welch)

In [6]:
w = df[(df.method == "welch") & df.est_bpm.notna() & df.ref_bpm.notna()].copy()
w["abs_err"] = (w.est_bpm - w.ref_bpm).abs()
worst = w.sort_values("abs_err", ascending=False).head(5)
worst[["session_id", "video_path", "window_start", "est_bpm", "ref_bpm", "abs_err", "quality_score"]].round(2)

,session_id,video_path,window_start,est_bpm,ref_bpm,abs_err,quality_score
2,1,data/raw/01_12bpm.mov,9.93,11.51,12.0,0.49,0.90
4,1,data/raw/01_12bpm.mov,19.87,12.26,12.0,0.26,0.95
5,1,data/raw/01_12bpm.mov,24.84,12.26,12.0,0.26,0.94
39,2,data/raw/02_20bpm.mov,0.00,19.75,20.0,0.25,0.92
43,2,data/raw/02_20bpm.mov,20.00,20.25,20.0,0.25,0.92


In [7]:
# Recompute the signal for the clips involved and plot the 3 worst windows
signals = {}
for sid, path in worst.head(3)[["session_id", "video_path"]].drop_duplicates().itertuples(index=False):
    signals[sid] = estimate("../" + path, "flow", ROI)

fig, axes = plt.subplots(3, 2, figsize=(12, 9))
for ax, r in zip(axes, worst.head(3).itertuples()):
    e = signals[r.session_id]; fs, win = e["fs"], e["window_s"]
    filt = bandpass(detrend(e["signal"]), e["low"], e["high"], fs)
    seg = filt[int(r.window_start * fs):int((r.window_start + win) * fs)]
    t = r.window_start + np.arange(len(seg)) / fs
    _, pk = peak_rate(e["signal"][int(r.window_start * fs):int((r.window_start + win) * fs)], fs, e["low"], e["high"])
    ax[0].plot(t, seg, lw=0.8); ax[0].plot(t[pk], seg[pk], "rv", ms=4)
    ax[0].set_title(f"{r.video_path.split('/')[-1]} @ {r.window_start:.0f}s: est {r.est_bpm:.1f} vs ref {r.ref_bpm:.0f} bpm")
    f = np.fft.rfftfreq(len(seg), 1 / fs) * 60; p = np.abs(np.fft.rfft(seg * np.hanning(len(seg)))) ** 2
    ax[1].plot(f, p, lw=0.8); ax[1].axvline(r.ref_bpm, color="g", ls="--", label="metronome")
    ax[1].axvline(r.est_bpm, color="r", ls=":", label="estimate"); ax[1].set_xlim(0, 40); ax[1].legend()
    ax[1].set_xlabel("breaths/min")
plt.tight_layout(); plt.savefig("../docs/figures/failure_worst_windows.png", dpi=130)

### 5b. Windows rejected by the quality gate

In [8]:
rej = df[(df.method == "welch") & df.est_bpm.isna()]
rej[["video_path", "window_start", "quality_score"]].round(2)

,video_path,window_start,quality_score
79,data/raw/03_hold.mov,5.0,0.58
80,data/raw/03_hold.mov,10.0,0.43
93,data/raw/04_wave.mov,0.0,0.53


## Findings (5 clips, 1 subject, metronome reference)

**Accuracy.** On the 26 scored windows from the two paced clips (12 and 20 breaths/min), every method was within +/-2 bpm 100% of the time. MAE: frame differencing 0.11, Welch (optical flow) 0.12, peak detection 0.18 breaths/min. Bias is within 0.04 bpm for all three and the Bland-Altman limits of agreement are about +/-0.4 bpm (peaks +/-0.5).

**Method comparison.** Welch and frame differencing are indistinguishable here; peak detection is slightly worse, mostly at 20 bpm (MAE 0.24 vs 0.13). These differences are about 0.1 bpm on 26 correlated windows from 2 clips, so they are not evidence that one method is better. The clips are clean, so this also says nothing about how frame differencing behaves in difficult conditions.

**Conditions.** Only the paced rate varies (12 and 20 bpm, both fine). Lighting, clothing, distance and camera mount are constant or unrecorded, so the per-condition tables cannot yet show where the method breaks.

**Uncertainty.** A bootstrap over subjects needs several subjects; with one it correctly reports "not estimable". The numbers above are point estimates for a single person.

**Failure analysis.** Among accepted windows the largest error is 0.49 bpm, and the three worst windows (all from the 12 bpm clip) show clean breathing waveforms with the spectral peak almost on the metronome rate. The error is far below the spectral bin width of the 30 s window (about 2 bpm), so these are not failures. The real failure cases are the ones the quality gate rejected: the breath-hold clip (2 of 5 windows, the ones containing the hold) and the hand-wave clip (its only window, quality 0.53 against a 0.6 threshold), where arm motion about 25x larger than breathing swamps the signal.

**Caveats.** The reference is the metronome target, not a measurement of actual breathing, so these numbers show agreement with a paced target for one person in one setup. They do not support any claim about accuracy in general, across people, lighting or clothing, or in unpaced breathing (the natural-breathing clip has no reference). An independent reference (accelerometer), varied conditions and more subjects are needed for that.